### 参数量对比

- 卷积神经网络---- in channel = 128 ; out channel = 64 ; C = 5*5 ; W = 25 * 2**13 = 204800
- 全连接层网络---- input = 4090 ; output = 1024 ; W = 4090 * 1024 = 4188160

### 小tips
- 一般我们称全连接神经网络为稠密网络（dense）或深度网络(deep) 
- 由于在卷积神经网络和循环神经网络中有权重共享的概念，所以一般来说相对于全连接神经网络，卷积神经网络和循环神经网络的权重数量都是非常少的

```mermaid
flowchart LR

    %% =========================
    %% RNN Cell
    %% =========================
    subgraph RNN["RNN Cell"]
        direction TB

        Xt["x_t"]
        Cell["RNN Cell"]
        Ht(("h_t"))

        Xt --> Cell
        Cell --> Ht

        %% 循环连接
        Cell -->|"Hidden State"| Cell
    end


    %% =========================
    %% Unrolled RNN
    %% =========================
    subgraph Unrolled["RNN Unrolled Through Time"]
        direction LR

        %% 初始隐藏状态
        H0(("h_0"))

        %% Time Step 1
        X1["x_1"]
        C1["RNN Cell"]
        H1(("h_1"))

        H0 --> C1
        X1 --> C1
        C1 --> H1

        %% Time Step 2
        X2["x_2"]
        C2["RNN Cell"]
        H2(("h_2"))

        H1 --> C2
        X2 --> C2
        C2 --> H2

        %% Time Step 3
        X3["x_3"]
        C3["RNN Cell"]
        H3(("h_3"))

        H2 --> C3
        X3 --> C3
        C3 --> H3

        %% Time Step 4
        X4["x_4"]
        C4["RNN Cell"]
        H4(("h_4"))

        H3 --> C4
        X4 --> C4
        C4 --> H4

        %% 后续时间步
        C4 --> More["..."]
    end


    %% =========================
    %% Styles
    %% =========================
    classDef hidden fill:#e8eefc,stroke:#344d7a,stroke-width:2px,color:#222;
    classDef input fill:#fff2cc,stroke:#98751b,stroke-width:2px,color:#222;
    classDef rnn fill:#e2efd9,stroke:#435b32,stroke-width:2px,color:#222;

    class Ht,H0,H1,H2,H3,H4 hidden;
    class Xt,X1,X2,X3,X4 input;
    class Cell,C1,C2,C3,C4 rnn;
```

### 1. 注意：一般来说循环神经网络适合处理一些与序列顺序有关的数据，比如自然语言
- rnn中可以将一个token理解为一个时间步，而一个时间步就会循环一次，所以在rnn中的输入向量中一般的维度是[Batch Size,Sequence Length,Feature Size]

```mermaid
flowchart LR
    H0["hₜ₋₁ ∈ ℝ⁽ʰⁱᵈᵈᵉⁿ_ˢⁱᶻᵉ⁾"]

    subgraph RNN["RNN Cell"]
        direction LR

        WHH["Wₕₕ hₜ₋₁ + bₕₕ"]
        ADD((+))
        TANH["tanh"]
        WIH["Wᵢₕ xₜ + bᵢₕ"]

        WHH --> ADD
        WIH --> ADD
        ADD --> TANH
    end

    X["xₜ ∈ ℝ⁽ⁱⁿᵖᵘᵗ_ˢⁱᶻᵉ⁾"]
    HT["hₜ ∈ ℝ⁽ʰⁱᵈᵈᵉⁿ_ˢⁱᶻᵉ⁾"]

    H0 --> WHH
    X --> WIH
    TANH --> HT

    style RNN fill:#e4f0dc,stroke:#385b2b,stroke-width:2px
    style WHH fill:#fff0cb,stroke:#a17b32
    style WIH fill:#fce5d8,stroke:#b77b50
    style ADD fill:#dce6f5,stroke:#526b9a
    style TANH fill:#dce6f5,stroke:#526b9a
```
RNN的隐藏状态是 $h_t=\tanh(W_{hh}h_{t-1}+b_{hh}+W_{ih}x_t+b_{ih})$


### 2. 使用RNNcell构建循环神经网络

In [ ]:
# 其中h是隐藏层的维度，而i是输入层的维度
import torch
import torch.nn as nn

inputdata = torch.tensor([1,2,3,4,5])
hidden = torch.tensor([1,2,3,4,5,6,7,8])
RNNcell = nn.RNNCell(input_size=5,hidden_size=8)
hidden = RNNcell(inputdata,hidden)
# inputdata of shape([batch_size,fearture_size])
# hidden of shape([batch_size,hidden_size])
# 如果使用cell的话我们需要构建dataset.shap([seqlen,batch_size,feature_size])

In [ ]:
# 示例
import torch

batch_size = 1
seq_len = 3
input_size = 4
hidden_size = 2

cell = torch.nn.RNNCell(input_size=input_size,hidden_size=hidden_size)

# (seq, batch, features)
dataset = torch.randn(seq_len, batch_size, input_size)
# 将初始隐层设置为0矩阵
hidden = torch.zeros(batch_size, hidden_size)

for idx, input in enumerate(dataset):
    print('=' * 20, idx, '=' * 20)
    print('Input size:', input.shape)

    hidden = cell(input, hidden)

    print('outputs size:', hidden.shape)
    print(hidden)

==================== 0 ====================
Input size: torch.Size([1, 4])
outputs size: torch.Size([1, 2])
tensor([[0.3865, 0.4651]], grad_fn=<TanhBackward0>)
==================== 1 ====================
Input size: torch.Size([1, 4])
outputs size: torch.Size([1, 2])
tensor([[0.3720, 0.1736]], grad_fn=<TanhBackward0>)
==================== 2 ====================
Input size: torch.Size([1, 4])
outputs size: torch.Size([1, 2])
tensor([[-0.0216,  0.4594]], grad_fn=<TanhBackward0>)


### 3. 直接使用RNN构建循环神经网络

```mermaid

flowchart LR

    %% 第一个时间步
    subgraph T1["时间步 1"]
        direction TB
        C1["RNN Cell"]
        B1["RNN Cell"]
        A1["RNN Cell"]
        C1 --> B1
        B1 --> A1
    end

    %% 第二个时间步
    subgraph T2["时间步 2"]
        direction TB
        C2["RNN Cell"]
        B2["RNN Cell"]
        A2["RNN Cell"]
        C2 --> B2
        B2 --> A2
    end

    %% 第三个时间步
    subgraph T3["时间步 3"]
        direction TB
        C3["RNN Cell"]
        B3["RNN Cell"]
        A3["RNN Cell"]
        C3 --> B3
        B3 --> A3
    end

    %% 第四个时间步
    subgraph T4["时间步 4"]
        direction TB
        C4["RNN Cell"]
        B4["RNN Cell"]
        A4["RNN Cell"]
        C4 --> B4
        B4 --> A4
    end

    %% 最后一个时间步
    subgraph TN["时间步 N"]
        direction TB
        CN["RNN Cell"]
        BN["RNN Cell"]
        AN["RNN Cell"]
        CN --> BN
        BN --> AN
    end

    %% 水平方向：时间传递
    A1 --> A2 --> A3 --> A4
    B1 --> B2 --> B3 --> B4
    C1 --> C2 --> C3 --> C4

    A4 --> D1["······"] --> AN
    B4 --> D2["······"] --> BN
    C4 --> D3["······"] --> CN

    %% 初始隐藏状态
    h01(("h₀¹")) --> A1
    h02(("h₀²")) --> B1
    h03(("h₀³")) --> C1

    %% 输入
    x1(("x₁")) --> A1
    x2(("x₂")) --> A2
    x3(("x₃")) --> A3
    x4(("x₄")) --> A4
    xN(("xₙ")) --> AN

    %% 最终隐藏状态
    AN --> hN1(("hₙ¹"))
    BN --> hN2(("hₙ²"))
    CN --> hN3(("hₙ³"))

    %% 颜色
    classDef l1 fill:#e0efd9,stroke:#526b45,stroke-width:2px,color:#344b2c;
    classDef l2 fill:#fff0c9,stroke:#98731c,stroke-width:2px,color:#765510;
    classDef l3 fill:#fce3d2,stroke:#96501f,stroke-width:2px,color:#87451b;
    classDef state fill:#e4edfc,stroke:#304c78,stroke-width:2px,color:#233b62;

    class A1,A2,A3,A4,AN l1;
    class B1,B2,B3,B4,BN l2;
    class C1,C2,C3,C4,CN l3;
    class h01,h02,h03,hN1,hN2,hN3 state;

```

In [ ]:
feature_size = 5
hidden_size = 8
num_layers = 2
seq_len = 7
inputs = torch.randn(seq_len,batch_size,feature_size)

RNNlayer = nn.RNN(input_size=input_size,hidden_size=hidden_size,num_layers=num_layers)
# 这里的numlayer是指在一个RNN神经网络中需要需要堆叠几次，即使用第一次计算后的隐层再一次进行计算，最后输出会增加numlayer个维度
out,hidden = RNNlayer(inputs,hidden)
# out 一般是一次循环中的所有隐藏层的值，hidden是最后一次隐藏层
# inputs of shape([seqlen,batch,feature_size])
# hidden of shape([num_layers,batch,hidden_size])
# out of shape([seqlen,batch,hidden_size])


<img src="./rnn.svg" width="100%">

In [2]:
# 代码示例
import torch
import torch.nn as nn
# 输入数据的特征值
batch_size = 1
input_size = 4
hidden_size = 2
num_layers = 3
seq_len = 7

# 构建数据集
inputsdata = torch.randn(seq_len,batch_size,input_size)
hidden = torch.zeros(num_layers,batch_size,hidden_size)

RNN = nn.RNN(input_size=input_size,hidden_size=hidden_size,num_layers=num_layers)

out,hidden = RNN(inputsdata,hidden)
print('Output size:', out.shape)
print('Output:', out)
print('Hidden size: ', hidden.shape)
print('Hidden: ', hidden)


Output size: torch.Size([7, 1, 2])
Output: tensor([[[ 0.0178, -0.6496]],

        [[ 0.1979, -0.6298]],

        [[ 0.2511, -0.5394]],

        [[ 0.4633, -0.4456]],

        [[ 0.2181, -0.4038]],

        [[ 0.5083, -0.4672]],

        [[ 0.4656, -0.3334]]], grad_fn=<StackBackward0>)
Hidden size:  torch.Size([3, 1, 2])
Hidden:  tensor([[[ 0.6793,  0.0128]],

        [[-0.6122,  0.7219]],

        [[ 0.4656, -0.3334]]], grad_fn=<StackBackward0>)


#### * RNN中的其他参数
- batch_first = True（将输入的数据集结构改为[batch_size,seq_len,input_size]）

#### * 一些常见的深度学习任务
| 任务 | 输入 | 输出 | 典型应用 |
|---|---|---|---|
| 分类 | 图片/文本 | 类别 | 猫狗识别 |
| 回归 | 特征数据 | 连续数值 | 房价预测 |
| 目标检测 | 图片 | 类别 + 边界框 | YOLO |
| 图像分割 | 图片 | 像素类别 | 医学图像分割 |
| 序列标注 | 序列 | 对应标签序列 | 命名实体识别 |
| Seq2Seq | 输入序列 | 输出序列 | 机器翻译 |
| 时间序列预测 | 历史序列 | 未来数值 | 天气预测 |
| 生成任务 | 随机噪声/条件 | 新数据 | GPT、Stable Diffusion |

### 4. rnn的例子

<img src="./one_hot.svg" width="100%">

#### 现在大语言模型的词处理
```mermaid
flowchart TD
    A["原始文本 Text<br/>I love deep learning"]
    B["Tokenizer 分词器<br/>将文本切分成 Token"]
    C["Token 序列<br/>I / love / deep / learning"]
    D["Vocabulary / Dictionary<br/>词汇表查询"]
    E["Token IDs<br/>[40, 3021, 1378, 6502]"]
    F["Embedding Layer<br/>将 Token ID 映射为稠密向量"]
    G["Embedding Vectors<br/>向量序列"]
    H["Positional Information<br/>位置信息处理"]
    I["Transformer<br/>Self-Attention + Feed Forward"]
    J["Contextual Representations<br/>上下文相关的向量表示"]
    K["LM Head + Softmax<br/>计算下一个 Token 的概率"]
    L["Output<br/>预测下一个 Token"]

    A --> B
    B --> C
    C --> D
    D --> E
    E --> F
    F --> G
    G --> H
    H --> I
    I --> J
    J --> K
    K --> L

    style A fill:#e1f0ff,stroke:#3572a5
    style E fill:#fff0cc,stroke:#b8860b
    style G fill:#dff2df,stroke:#4b8b4b
    style I fill:#ffe5cc,stroke:#c47b32
    style L fill:#eadcf8,stroke:#8655aa
```